In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers

import numpy as np
import ast

In [ ]:
all_times_df = pd.read_csv('predictors_plus_target_calibrated.csv')
all_times_df.head()

In [ ]:
all_times_df = all_times_df.drop(['p6177_i0', 'p90004', 'p4079_i0_a0', 'p94_i0_a0', 'p4080_i0_a0', 'p93_i0_a0', 'p20002_i0_a0', 'p6177_i0', 'p30760_i0', 'p30870_i0'], axis=1)
all_times_df.info()

In [ ]:
all_times_df = all_times_df.dropna()
all_times_df

In [ ]:
label_encoder = LabelEncoder()
all_times_df['activity_class'] = label_encoder.fit_transform(all_times_df['activity_class'])

all_times_df['p31'] = pd.factorize(all_times_df['p31'])[0]
all_times_df['p2306_i0'] = pd.factorize(all_times_df['p2306_i0'])[0]
all_times_df['p2443_i0'] = pd.factorize(all_times_df['p2443_i0'])[0]

"""all_times_df['MAD'] = all_times_df['MAD'] .apply(lambda x: np.fromstring(x.strip('[]'), sep=' '))
all_times_df['MAD'] = all_times_df['MAD'].apply(np.sum)

all_times_df['MPD'] = all_times_df['MPD'] .apply(lambda x: np.fromstring(x.strip('[]'), sep=' '))
all_times_df['MPD'] = all_times_df['MPD'].apply(np.sum)"""

In [ ]:
all_times_df

In [ ]:
X = all_times_df[['Dominant frequency', 'Peak power',  'PPFd', 'Entropy', 'PF_sum', 'p31', 'p21022', 'Mean', 'STD', 'Steps']]
y = all_times_df['activity_class']
X

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('First few labels:', y_train[:5])

In [ ]:
model = Sequential([
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.02)),
    Dropout(0.5),
    Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.02)),
    Dropout(0.5),
    Dense(3, activation='softmax') 
])

In [ ]:


model.compile(optimizer='Adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
checkpoint_callback = ModelCheckpoint('best_model.keras', 
                                     monitor='val_loss', 
                                     save_best_only=True, 
                                     mode='min', 
                                     verbose=1)

early_stopping_callback = EarlyStopping(monitor='val_loss', 
                                        patience=30,  
                                        restore_best_weights=True,
                                        verbose=1)

In [ ]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
history = model.fit(X_train_scaled, y_train, 
                    epochs=500,        
                    batch_size=64, 
                    validation_data=(X_test_scaled, y_test), 
                    callbacks=[checkpoint_callback, early_stopping_callback]) 

In [ ]:
best_model = keras.models.load_model('best_model.keras')

In [ ]:
test_loss, test_accuracy = best_model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
y_pred = best_model.predict(X_test_scaled)
y_pred_classes = tf.argmax(y_pred, axis=1).numpy()

cm = confusion_matrix(y_test, y_pred_classes)
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.xticks(rotation=45)
plt.yticks(rotation=0)  

plt.tight_layout() 
plt.show()